# Install packages

In [1]:
import json
import pandas as pd
from pandas import json_normalize
import ast  # for safely evaluating strings as Python literals if needed
import os
from ast import literal_eval
import numpy as np

In [2]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing 

In [3]:
# Reset ALL display options to default
pd.reset_option("^display.", silent=True)  # Resets all display-related options

# Import data

In [4]:
df = pd.read_csv('/kaggle/input/tmdbmoviedata2000-2024/alldata.csv') 
print(df.sample(10))

             id                                              title  \
244850  1352954                      Once Upon A Time In Indochine   
150820   484706            The Wiggles - Wiggly, Wiggly Christmas!   
234542  1102113                              Un bosque en silencio   
210543   919570                                             R.M.N.   
21790    709590                                   I Was Born, But…   
179236   614428                                             Kairos   
9654     288953  Forget Baghdad: Jews and Arabs - The Iraqi Con...   
114789   259855  Paycheck to Paycheck: The Life & Times of Katr...   
34341    127649                         Princess in an Iron Helmet   
256374  1014895                                  Maya and the Wave   

                              production_countries  revenue  \
244850                                          []        0   
150820                                          []        0   
234542                               ['A

In [5]:
len(df)

256778

# Primary release date - only keep movies released 2000-2024

In [6]:
df['primary_release_date'] = pd.to_datetime(df['primary_release_date'])

In [7]:
df['year'] = df['primary_release_date'].dt.year
df = df[(df['year'] >= 2000) & (df['year'] <= 2024)]
df['year'].value_counts()

year
2024    16119
2023    16018
2022    15067
2019    14998
2018    14426
2021    13885
2017    13733
2016    12550
2020    12525
2015    11846
2014    11407
2013    10935
2012    10005
2011     9326
2009     8778
2010     8565
2008     8317
2007     7937
2006     7487
2005     6749
2004     6139
2003     5659
2002     5134
2001     4766
2000     4401
Name: count, dtype: int64

# Remove movies with runtime <60 - not eligible for analysis

In [8]:
short_runtime_movies = df[df['runtime'] < 60]
print(f"\nNumber of movies with runtime < 60 minutes: {len(short_runtime_movies)}")

if len(short_runtime_movies) > 0:
    print("\nMovies with runtime < 60 minutes:")
    print(short_runtime_movies[['id', 'title', 'runtime']])
    print("\nRuntime values under 60 minutes:")
    print(short_runtime_movies['runtime'].unique())


Number of movies with runtime < 60 minutes: 34102

Movies with runtime < 60 minutes:
             id                               title  runtime
1       1468524                            黑色风景里的女人        0
11      1445598                                限期结婚        0
13      1432570     Sulla spiaggia e di là dal molo        0
15      1430035  Genesis – Six Hours Live 1972-1980        0
21      1422186                   Cuadro sangriento        0
...         ...                                 ...      ...
256740  1277881                      Aitana Bonmatí        0
256744  1376187                  Abaixo das Árvores        0
256747  1376195           La verdad del caso Yéremi        0
256770  1310704                            Cin Köyü        0
256771  1409009                       O Rei do Riso        0

[34102 rows x 3 columns]

Runtime values under 60 minutes:
[ 0 45 50 51 24 57 55 46 44 59 25 43 52 11 58 23 30 42 53 16 28 47 40 36
 32 49 41 56 10 48 35 54  7 13 26  8  1 21 27 20 

In [9]:
# Filter out movies with runtime < 60 minutes
df = df[df['runtime'] >= 60]

# Verify the removal by checking the minimum runtime
print(f"Minimum runtime after removal: {df['runtime'].min()}")
print(f"Number of movies remaining: {len(df)}")

Minimum runtime after removal: 60
Number of movies remaining: 222670


# Remove TV movies - not eligible for analysis

In [10]:
# First ensure genres column contains lists (not strings)
if isinstance(df['genres'].iloc[0], str):
    df['genres'] = df['genres'].apply(ast.literal_eval)

# Filter out movies with 'TV Movie' in their genres
df = df[~df['genres'].apply(lambda x: 'TV Movie' in x if isinstance(x, list) else False)]

# Reset index if needed (optional)
df = df.reset_index(drop=True)
print(f"Number of movies after removing 'TV Movie' genre: {len(df)}")

Number of movies after removing 'TV Movie' genre: 213123


In [11]:
# Check if any movies have 'TV Movie' genre
df['has_tv_movie'] = df['genres'].apply(lambda x: 'TV Movie' in x if isinstance(x, list) else False)

any_tv_movie = df['has_tv_movie'].any()
print(f"\nAny movies with 'TV Movie' genre: {any_tv_movie}")

# Show movies with TV Movie genre if any
if any_tv_movie:
    print("\nMovies with 'TV Movie' genre:")
    print(df[df['has_tv_movie']][['id', 'title', 'genres']])


Any movies with 'TV Movie' genre: False


# Remove movies that were only released on TV AND movies only had premiered - not eligible for analysis

In [12]:
import ast

def should_remove_movie(release_data):
    """Returns True if movie should be removed (exclusively type 1 or 6, no 2/3/4/5)."""
    # Convert string to dict if needed
    if isinstance(release_data, str):
        try:
            release_data = ast.literal_eval(release_data)
        except:
            return False  # Invalid format (keep the row)
    
    if not isinstance(release_data, dict):
        return False  # Keep invalid/missing data
    
    types = set()
    for country_releases in release_data.values():
        for release in country_releases:
            t = release.get('type')
            # Handle both string and integer types
            if isinstance(t, str) and t.isdigit():
                t = int(t)
            if isinstance(t, int):
                types.add(t)
    
    # Remove movies with ONLY type 1, 6, or both
    return types.issubset({1, 6}) and len(types) > 0

# Split into kept/removed movies
valid_movies = df[~df['released_countries'].apply(should_remove_movie)]
removed_movies = df[df['released_countries'].apply(should_remove_movie)]

In [13]:
# Overwrite DataFrame
df = valid_movies.copy()

# Show remaining counts
print(f"\nRemaining movies: {len(df)}")
print(f"Removed movies: {len(removed_movies)}")


Remaining movies: 181746
Removed movies: 31377


# Remove movies that were released only physically (DVD/Bluerays) or only digitally 

In [14]:
import ast

def should_remove_movie(release_data):
    """Returns True if movie should be removed (exclusively type 1 or 6, no 2/3/4/5)."""
    # Convert string to dict if needed
    if isinstance(release_data, str):
        try:
            release_data = ast.literal_eval(release_data)
        except:
            return False  # Invalid format (keep the row)
    
    if not isinstance(release_data, dict):
        return False  # Keep invalid/missing data
    
    types = set()
    for country_releases in release_data.values():
        for release in country_releases:
            t = release.get('type')
            # Handle both string and integer types
            if isinstance(t, str) and t.isdigit():
                t = int(t)
            if isinstance(t, int):
                types.add(t)
    
    # Remove movies with ONLY type 1, 6, or both
    return types.issubset({5, 4}) and len(types) > 0

# Split into kept/removed movies
valid_movies = df[~df['released_countries'].apply(should_remove_movie)]
removed_movies = df[df['released_countries'].apply(should_remove_movie)]

In [15]:
# Overwrite DataFrame
df = valid_movies.copy()

# Show remaining counts
print(f"\nRemaining movies: {len(df)}")
print(f"Removed movies: {len(removed_movies)}")


Remaining movies: 149673
Removed movies: 32073


# Define North/South

## Show the unique countries or terrotires of the data

In [16]:
from ast import literal_eval
from collections import Counter

# Convert string representations of lists to actual lists (if needed)
production_countries = df['production_countries'].apply(
    lambda x: literal_eval(x) if isinstance(x, str) else x
)

# Count country frequencies (including those in multi-country entries)
country_counter = Counter()
for countries_list in production_countries:
    country_counter.update(countries_list)  # Count each country in the list

# Get the total number of unique countries/territories
num_unique_countries = len(country_counter)
print(f"Number of unique countries or territories: {num_unique_countries}\n")

# Sort countries by frequency (most to least)
most_common_countries = country_counter.most_common()

# Print results
print("Country Frequencies (Most to Least Common):")
for country, count in most_common_countries:
    print(f"{country}: {count}")

Number of unique countries or territories: 230

Country Frequencies (Most to Least Common):
United States of America: 29652
India: 9684
France: 8710
Japan: 8188
United Kingdom: 7466
Germany: 6888
Canada: 4711
China: 3690
South Korea: 3595
Spain: 3533
Italy: 3494
Russia: 2838
Brazil: 2748
Argentina: 2487
Mexico: 2411
Belgium: 1960
Indonesia: 1793
Philippines: 1791
Netherlands: 1767
Hong Kong: 1677
Australia: 1646
Sweden: 1606
Thailand: 1286
Switzerland: 1264
Denmark: 1205
Poland: 1158
Turkey: 1096
Austria: 1044
Iran: 1023
Norway: 973
Czech Republic: 953
Finland: 925
Taiwan: 804
Portugal: 784
Egypt: 779
Ireland: 776
Israel: 727
Malaysia: 726
Greece: 712
Chile: 710
Serbia: 568
Romania: 554
Hungary: 550
Ukraine: 541
South Africa: 537
Colombia: 502
Croatia: 478
New Zealand: 427
Kazakhstan: 393
Nigeria: 388
Slovenia: 380
Peru: 366
Bulgaria: 347
Luxembourg: 346
Mongolia: 344
Uruguay: 341
Estonia: 336
Slovakia: 329
Singapore: 329
Latvia: 325
Iceland: 281
Lithuania: 278
Venezuela: 274
Puerto Ri

## Define north/south

In [17]:
# Step 1: Safely read and convert production_countries if they're strings
df['production_countries'] = df['production_countries'].apply(
    lambda x: literal_eval(x) if isinstance(x, str) else x
)

# Step 2: Read north countries (with error handling)
with open('/kaggle/input/northerncountries/north.txt', 'r') as f:
    north_countries = [line.strip() for line in f if line.strip()]  # skip empty lines

# Step 3: Enhanced categorization
def categorize_countries(countries_list):
    if not countries_list or countries_list == ['']:
        return 'unknown'
    
    # Handle cases where country names might differ
    matches = [c in north_countries for c in countries_list]
    
    if all(matches):
        return 'north'
    elif any(matches):
        return 'coproduction'
    else:
        return 'south'

# Step 4: Apply function
df['region'] = df['production_countries'].apply(categorize_countries)

# Verification
print(df['region'].value_counts())
print("\nSample entries:")
print(df[['production_countries', 'region']].head(10))

region
north           78433
south           33618
unknown         32111
coproduction     5511
Name: count, dtype: int64

Sample entries:
          production_countries   region
2                           []  unknown
3                           []  unknown
4                           []  unknown
6                           []  unknown
7                           []  unknown
10  [United States of America]    north
11  [United States of America]    north
15                          []  unknown
22                          []  unknown
23                          []  unknown


In [18]:
# Check if the categorization worked
df.loc[df['region'] == 'unknown', 'production_countries'].value_counts()

production_countries
[]    32111
Name: count, dtype: int64

In [19]:
df.loc[df['region'] == 'coproduction', 'production_countries'].value_counts()

production_countries
[Argentina, Spain]                             119
[Mexico, United States of America]             115
[India, United States of America]               86
[China, United States of America]               85
[Mexico, Spain]                                 43
                                              ... 
[Argentina, France, Mexico]                      1
[Denmark, Pakistan]                              1
[Australia, Germany, Kenya, United Kingdom]      1
[Slovakia, Czech Republic, Mexico]               1
[China, Finland]                                 1
Name: count, Length: 2916, dtype: int64

In [20]:
df.loc[df['region'] == 'south', 'production_countries'].value_counts()

production_countries
[India]                               9199
[China]                               2776
[Brazil]                              2234
[Mexico]                              1791
[Indonesia]                           1687
                                      ... 
[Indonesia, Philippines]                 1
[New Caledonia]                          1
[Argentina, Chile, Uruguay, China]       1
[Argentina, Cuba, Mexico, Peru]          1
[Costa Rica, Guatemala, Mexico]          1
Name: count, Length: 550, dtype: int64

In [21]:
df.loc[df['region'] == 'north', 'production_countries'].value_counts()

production_countries
[United States of America]                                              23499
[Japan]                                                                  7511
[United Kingdom]                                                         4127
[France]                                                                 3954
[Germany]                                                                3513
                                                                        ...  
[Belgium, Ireland, Netherlands]                                             1
[Czech Republic, Israel, Poland, Slovakia, United States of America]        1
[Poland, Serbia]                                                            1
[Poland, Romania, Italy]                                                    1
[Albania, Bulgaria, Israel, United States of America]                       1
Name: count, Length: 2861, dtype: int64

# Define missing values

In [22]:
df.isna().sum()

id                           0
title                        0
production_countries         0
revenue                      0
primary_release_date         0
released_countries           0
overview                  4311
tagline                 108514
runtime                      0
budget                       0
genres                       0
popularity                   0
original_language            0
keywords                     0
watch_providers              0
imdb_id                  22016
year                         0
has_tv_movie                 0
region                       0
dtype: int64

In [23]:
df.head()

,id,title,production_countries,revenue,primary_release_date,released_countries,overview,tagline,runtime,budget,genres,popularity,original_language,keywords,watch_providers,imdb_id,year,has_tv_movie,region
2,1464523,Seven Ages: The Story of the Irish State,[],0,2000-01-01,{'IE': [{'type': 3}]},"A documentary series charting the birth, growt...",NaN,385,0,[],0.0000,en,[],{},NaN,2000,False,unknown
3,1458037,Aakhir Kaun Thi Woh?,[],0,2000-01-01,{'IN': [{'type': 3}]},The film was shot and ready to release in 1990...,NaN,110,0,[],0.0643,hi,[],{},NaN,2000,False,unknown
4,1458029,Ganga Dacait,[],0,2000-01-01,{'IN': [{'type': 3}]},"GENRE: Action STAR CAST: Sapna Sappu, Amit Pa...",NaN,135,0,[],0.0898,hi,[],{},NaN,2000,False,unknown
6,1453652,Basanti,[],0,2000-01-01,{'IN': [{'type': 3}]},Basanti is a brave working class teenage girl ...,NaN,110,0,[],0.2139,hi,[],{},NaN,2000,False,unknown
7,1453430,Daku Rani,[],0,2000-01-01,{'IN': [{'type': 3}]},"Starring : Kiran Kumar,Anil Nagrath,Durgesh Na...",NaN,106,0,[],0.0785,hi,[],{},NaN,2000,False,unknown


In [24]:
for col in ['keywords', 'watch_providers','genres','original_language','released_countries']:
    df[col] = df[col].apply(lambda x: None if x == [] else x)

for col in ['revenue', 'runtime', 'budget']:
    df[col] = df[col].replace(0, np.nan)

def clean_field(x):
    if x is None or (isinstance(x, str) and x.strip() == ""):
        return np.nan
    return x

df['imdb_id'] = df['imdb_id'].apply(clean_field)
df['tagline'] = df['tagline'].apply(clean_field)
df['overview'] = df['overview'].apply(clean_field)

In [25]:
df.isnull().sum()

id                           0
title                        0
production_countries         0
revenue                 136163
primary_release_date         0
released_countries           0
overview                  4312
tagline                 108514
runtime                      0
budget                  132495
genres                   14550
popularity                   0
original_language            0
keywords                     0
watch_providers              0
imdb_id                  22016
year                         0
has_tv_movie                 0
region                       0
dtype: int64

In [26]:
columns_to_check = ['revenue', 'overview', 'tagline', 'budget', 'genres', 'imdb_id']

In [27]:
# Group by 'production_region' and compute missing values (count and percentage)
missing_stats = df.groupby('region')[columns_to_check].agg(
    # For each column, compute two metrics:
    # 1. Total missing values (count)
    # 2. Percentage of missing values (relative to group size)
    **{
        f"{col}_missing": (col, lambda x: x.isna().sum()) for col in columns_to_check
    },
    **{
        f"{col}_missing_pct": (col, lambda x: x.isna().mean() * 100) for col in columns_to_check
    }
)

In [28]:
# Reorder columns to group counts and percentages together
new_columns = []
for col in columns_to_check:
    new_columns.extend([f"{col}_missing", f"{col}_missing_pct"])

missing_stats = missing_stats[new_columns]

In [29]:
missing_stats.style.format("{:.2f}", subset=pd.IndexSlice[:, [c for c in missing_stats.columns if 'pct' in c]])

,revenue_missing,revenue_missing_pct,overview_missing,overview_missing_pct,tagline_missing,tagline_missing_pct,budget_missing,budget_missing_pct,genres_missing,genres_missing_pct,imdb_id_missing,imdb_id_missing_pct
region,,,,,,,,,,,,
coproduction,4648,84.34,49,0.89,3866,70.15,4657,84.50,134,2.43,181,3.28
north,68390,87.20,2005,2.56,51641,65.84,66298,84.53,2684,3.42,6727,8.58
south,31346,93.24,1320,3.93,27578,82.03,31219,92.86,2218,6.60,5775,17.18
unknown,31779,98.97,938,2.92,25429,79.19,30321,94.43,9514,29.63,9333,29.06


# Topic extraction

## preprocessing

In [30]:
df.isna().sum()

id                           0
title                        0
production_countries         0
revenue                 136163
primary_release_date         0
released_countries           0
overview                  4312
tagline                 108514
runtime                      0
budget                  132495
genres                   14550
popularity                   0
original_language            0
keywords                     0
watch_providers              0
imdb_id                  22016
year                         0
has_tv_movie                 0
region                       0
dtype: int64

In [31]:
bert_df = df.copy()
bert_df['genres'] = bert_df['genres'].astype(str)
bert_df['keywords'] = bert_df['keywords'].astype(str)
bert_df = bert_df.dropna(subset=['overview'])
def build_full_description(row):
    parts = []
    if pd.notnull(row['title']):
        parts.append(row['title'])
    if pd.notnull(row['tagline']):
        parts.append(row['tagline'])
    if pd.notnull(row['keywords']):
        parts.append(row['keywords'])
    if pd.notnull(row['overview']):
        parts.append(row['overview'])
    return " | ".join(parts)

bert_df['full_description'] = bert_df.apply(build_full_description, axis=1)

In [32]:
#clean text
bert_df['full_description'] = bert_df['full_description'].str.replace(r'\n', ' ', regex=True)
bert_df['full_description'] = bert_df['full_description'].str.replace(r'\s*\|\s*', ' | ', regex=True)

In [33]:
#check missing
bert_df['full_description'].isna().sum()

0

In [34]:
texts = bert_df['full_description'].tolist()
titles= bert_df['title'].tolist()

In [35]:
texts[0]

'Seven Ages: The Story of the Irish State | [] | A documentary series charting the birth, growth and development of the Irish state from its foundation in 1921 until the 1990s.'

In [36]:
titles[0]

'Seven Ages: The Story of the Irish State'

In [37]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(texts, show_progress_bar=True)

2025-05-25 22:03:36.635093: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748210616.808678      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748210616.861974      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4543 [00:00<?, ?it/s]

In [38]:
# Preventing Stochastic Behavior
from umap import UMAP

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

In [48]:
# Controling number of topics
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_cluster_size=60,metric='euclidean', cluster_selection_method='eom', prediction_data=True) #Larger min_cluster_size → fewer, broader clusters (avoids overfitting to small/noisy groups).

In [40]:
# import default representation
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

In [41]:
#additional representations
import openai
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech

# KeyBERT
keybert_model = KeyBERTInspired() #basicrepresentations that increases coherence and reduce stopwords

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# GPT-3.5
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
topic: <topic label>
"""
client = openai.OpenAI(api_key="sk-...")
openai_model = OpenAI(client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    "POS": pos_model
}

## Training

In [49]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
    calculate_probabilities=True,
  verbose=True
)

topics, probs = topic_model.fit_transform(texts, embeddings)

topic_model.get_topic_info()

2025-05-25 22:13:54,660 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-25 22:17:14,866 - BERTopic - Dimensionality - Completed ✓
2025-05-25 22:17:14,873 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-25 22:20:43,201 - BERTopic - Cluster - Completed ✓
2025-05-25 22:20:43,229 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-25 22:22:03,251 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,73803,-1_life_love_woman_young,"[life, love, woman, young, world, new, story, ...","[film, movie, relationship, sex, comedy, woman...","[woman, world, family, film, relationship, liv...","[life, love, woman, young, world, new, story, ...",[Curtiz | No country should change a man's cha...
1,0,11707,0_bollywood_love_india_village,"[bollywood, love, india, village, film, indian...","[bollywood, krishna, kumar, raj, khan, arjun, ...","[bollywood, village, indian, falls love, movie...","[bollywood, love, village, film, indian, story...",[Jo Bole So Nihaal | ['bollywood'] | Jo Bole S...
2,1,6187,1_el_la_spanish_brazilian,"[el, la, spanish, brazilian, brazil, mexico, a...","[spanish civil, juan, franco, spain, mexican f...","[el, brazilian, mexico, argentina, documentary...","[spanish, brazilian, documentary, mexican, del...","[El joven Berlanga | ['madrid, spain', 'cinema..."
3,2,4984,2_concert_live_band_music,"[concert, live, band, music, rock, tour, album...","[live concert, concert, concert film, concerts...","[concert, band, tour, songs, rock roll, concer...","[concert, live, band, music, rock, tour, album...",[NAMI TAMAKI 2nd CONCERT Make Progress ~road t...
4,3,3169,3_killer_police_murder_detective,"[killer, police, murder, detective, bank, crim...","[serial killer, murders, homicide, detective, ...","[detective, cop, serial killer, robbery, crimi...","[killer, police, murder, detective, bank, crim...",[Killer Me | A journey through the mind of a s...
...,...,...,...,...,...,...,...,...
163,162,66,162_grace_grace lee_saving grace_grace grace,"[grace, grace lee, saving grace, grace grace, ...","[life grace, grace grace, grace love, love gra...","[grace, grace lee, saving grace, grace grace, ...","[grace, life, funeral home, forgiveness, funer...",[An American Girl: Grace Stirs Up Success | []...
164,163,66,163_phone_caller_calls_cell phone,"[phone, caller, calls, cell phone, cell, booth...","[cell phone, cellphone, mysterious phone, tele...","[caller, cell phone, phone booth, telemarketer...","[phone, caller, calls, cell, booth, killer, nu...",[The President's Cell Phone | [] | An old man ...
165,164,63,164_et_une_dans_est,"[et, une, dans, est, le, la, sa, il, qu, elle]","[dans une, dans le, une, dans la, qu il, qu il...","[et, une, dans, est, le, les, des, pas, vie, q...","[et, une, dans, est, le, elle, pour, qui, lui,...",[Tough Luck | [] | Lorsque vous rencontrez Réj...
166,165,63,165_greece_greek_athens_theo,"[greece, greek, athens, theo, athenian, crisis...","[live greek, growing greek, greeks, greek, gre...","[greece, greek, athens, theo, athenian, athens...","[greece, crisis, και, island, economic crisis,...",[Angelo Tsarouchas - It's All Greek to Me | Th...


In [50]:
topic_model.get_topic(1, full=True)

{'Main': [('el', 0.01010281334679044),
  ('la', 0.008998867025727567),
  ('spanish', 0.0067788050592046505),
  ('brazilian', 0.006705911783699842),
  ('brazil', 0.006540060606987233),
  ('mexico', 0.006043682082760663),
  ('argentina', 0.005335055300575734),
  ('spain', 0.0052264082662239995),
  ('documentary', 0.005029311714053426),
  ('mexican', 0.00492019619797845)],
 'KeyBERT': [('spanish civil', 0.5424948),
  ('juan', 0.4531927),
  ('franco', 0.4505022),
  ('spain', 0.44002324),
  ('mexican feature', 0.4397728),
  ('spanish', 0.4360562),
  ('miguel', 0.42287323),
  ('mexican', 0.4221406),
  ('carlos', 0.41866913),
  ('mexico', 0.4184133)],
 'MMR': [('el', 0.01010281334679044),
  ('brazilian', 0.006705911783699842),
  ('mexico', 0.006043682082760663),
  ('argentina', 0.005335055300575734),
  ('documentary', 0.005029311714053426),
  ('rio', 0.004191336829724885),
  ('dictatorship', 0.0038430058287722427),
  ('buenos', 0.0036183906214124584),
  ('buenos aires', 0.0035950458059596196)

In [62]:
# Reduce outliers
new_topics = topic_model.reduce_outliers(texts, topics)

# Reduce outliers with pre-calculate embeddings instead
new_topics = topic_model.reduce_outliers(texts, topics, strategy="embeddings", embeddings=embeddings)

100%|██████████| 74/74 [01:02<00:00,  1.18it/s]


In [66]:
topic_model.update_topics(texts, topics=new_topics)

2025-05-25 22:30:16,008 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [65]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,0,13085,0_bollywood_his_is_he,"[bollywood, his, is, he, love, to, who, her, a...","[bollywood, krishna, kumar, raj, khan, arjun, ...","[bollywood, village, indian, falls love, movie...","[bollywood, love, village, film, indian, story...",[Jo Bole So Nihaal | ['bollywood'] | Jo Bole S...
1,1,7708,1_de_el_la_of,"[de, el, la, of, in, the, and, his, that, to]","[spanish civil, juan, franco, spain, mexican f...","[el, brazilian, mexico, argentina, documentary...","[spanish, brazilian, documentary, mexican, del...","[El joven Berlanga | ['madrid, spain', 'cinema..."
2,2,6116,2_concert_music_band_live,"[concert, music, band, live, rock, tour, album...","[live concert, concert, concert film, concerts...","[concert, band, tour, songs, rock roll, concer...","[concert, live, band, music, rock, tour, album...",[NAMI TAMAKI 2nd CONCERT Make Progress ~road t...
3,3,7173,3_killer_murder_police_crime,"[killer, murder, police, crime, serial, detect...","[serial killer, murders, homicide, detective, ...","[detective, cop, serial killer, robbery, crimi...","[killer, police, murder, detective, bank, crim...",[Killer Me | A journey through the mind of a s...
4,4,4668,4_ghost_horror_haunted_house,"[ghost, horror, haunted, house, evil, paranorm...","[haunted, ghost story, ghosts, haunted house, ...","[haunted, paranormal, supernatural, ghosts, ha...","[ghost, haunted, horror, house, devil, paranor...",[Trico Tri Happy Halloween | There's a party a...
...,...,...,...,...,...,...,...,...
162,162,231,162_grace_her_she_life,"[grace, her, she, life, gloria, to, is, and, w...","[life grace, grace grace, grace love, love gra...","[grace, grace lee, saving grace, grace grace, ...","[grace, life, funeral home, forgiveness, funer...",[An American Girl: Grace Stirs Up Success | []...
163,163,290,163_phone_call_cell_calls,"[phone, call, cell, calls, caller, her, number...","[cell phone, cellphone, mysterious phone, tele...","[caller, cell phone, phone booth, telemarketer...","[phone, caller, calls, cell, booth, killer, nu...",[The President's Cell Phone | [] | An old man ...
164,164,138,164_de_un_et_une,"[de, un, et, une, le, dans, la, est, les, pour]","[dans une, dans le, une, dans la, qu il, qu il...","[et, une, dans, est, le, les, des, pas, vie, q...","[et, une, dans, est, le, elle, pour, qui, lui,...",[Tough Luck | [] | Lorsque vous rencontrez Réj...
165,165,431,165_greek_greece_athens_his,"[greek, greece, athens, his, he, in, to, the, ...","[live greek, growing greek, greeks, greek, gre...","[greece, greek, athens, theo, athenian, athens...","[greece, crisis, και, island, economic crisis,...",[Angelo Tsarouchas - It's All Greek to Me | Th...


In [68]:
# Get detailed topic info
topic_info = topic_model.get_topic_info()
topic_info.to_csv("topic_descriptions.csv", index=False)

## Visualize

In [69]:
topic_model.visualize_topics(custom_labels=True)

In [70]:
topic_model.visualize_hierarchy(custom_labels=True)

In [71]:
topic_model.visualize_distribution(probs[200])

## Visualize documents

In [54]:
import itertools
import pandas as pd

# Define colors for the visualization to iterate over
colors = itertools.cycle(['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe', '#008080', '#e6beff', '#9a6324', '#fffac8', '#800000', '#aaffc3', '#808000', '#ffd8b1', '#000075', '#808080', '#ffffff', '#000000'])
color_key = {str(topic): next(colors) for topic in set(topic_model.topics_) if topic != -1}

# Prepare dataframe and ignore outliers
df = pd.DataFrame({"x": reduced_embeddings_2d[:, 0], "y": reduced_embeddings_2d[:, 1], "Topic": [str(t) for t in topic_model.topics_]})
df["Length"] = [len(doc) for doc in docs]
df = df.loc[df.Topic != "-1"]
df = df.loc[(df.y > -10) & (df.y < 10) & (df.x < 10) & (df.x > -10), :]
df["Topic"] = df["Topic"].astype("category")

# Get centroids of clusters
mean_df = df.groupby("Topic").mean().reset_index()
mean_df.Topic = mean_df.Topic.astype(int)
mean_df = mean_df.sort_values("Topic")

NameError: name 'reduced_embeddings_2d' is not defined

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
from adjustText import adjust_text
import matplotlib.patheffects as pe

fig = plt.figure(figsize=(16, 16))
sns.scatterplot(data=df, x='x', y='y', c=df['Topic'].map(color_key), alpha=0.4, sizes=(0.4, 10), size="Length")

# Annotate top 50 topics
texts, xs, ys = [], [], []
for row in mean_df.iterrows():
  topic = row[1]["Topic"]
  name = " - ".join(list(zip(*topic_model.get_topic(int(topic))))[0][:3])

  if int(topic) <= 50:
    xs.append(row[1]["x"])
    ys.append(row[1]["y"])
    texts.append(plt.text(row[1]["x"], row[1]["y"], name, size=10, ha="center", color=color_key[str(int(topic))],
                          path_effects=[pe.withStroke(linewidth=0.5, foreground="black")]))

# Adjust annotations such that they do not overlap
adjust_text(texts, x=xs, y=ys, time_lim=1, force_text=(0.01, 0.02), force_static=(0.01, 0.02), force_pull=(0.5, 0.5))
plt.show()
# plt.savefig("visualization2.png", dpi=600)

In [ ]:
pip install datamapplot

In [ ]:
import datamapplot
import re

# Create a label for each document
llm_labels = [re.sub(r'\W+', ' ', label[0][0].split("\n")[0].replace('"', '')) for label in topic_model.get_topics(full=True)["LLM"].values()]
llm_labels = [label if label else "Unlabelled" for label in llm_labels]
all_labels = [llm_labels[topic+topic_model._outliers] if topic != -1 else "Unlabelled" for topic in topics]

# Run the visualization
datamapplot.create_plot(
    reduced_embeddings,
    all_labels,
    label_font_size=11,
    title="ArXiv - BERTopic",
    sub_title="Topics labeled with `openhermes-2.5-mistral-7b`",
    label_wrap_width=20,
    use_medoids=True,
    logo=bertopic_logo,
    logo_width=0.16
)

## Topics overtime

In [78]:
timestamps=bert_df['year'].to_list()
topics_over_time = topic_model.topics_over_time(docs=texts, 
                                                timestamps=timestamps, 
                                                global_tuning=True, 
                                                evolution_tuning=True, 
                                                nr_bins=20)

20it [01:31,  4.58s/it]


In [79]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=20)

## Topics per region

In [81]:
classes = bert_df["region"]

In [83]:
topics_per_class = topic_model.topics_per_class(texts, topics, classes=classes)

TypeError: BERTopic.topics_per_class() got multiple values for argument 'classes'

In [ ]:
topic_model.visualize_topics_per_class(topics_per_class, top_n_topics=10, width=900)

# Save model

In [84]:
# Save model
topic_model.save("my_model")

2025-05-25 22:52:22,176 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [ ]:
# Load model
my_model = BERTopic.load("my_model")